# Bubbleless Phase 1 (v0.1) — Colab T4 异构气泡检测与首尾重叠实验

**使用前请确认**：菜单栏 `代码执行程序 (Runtime)` -> `更改运行时类型 (Change runtime type)` -> 选择 **`T4 GPU`**。

本实验在真实 **NVIDIA T4 (15GB 显存 + 2核 CPU + 12.7GB 内存)** 上验证：
1. **三组件探针（IO / CPU / GPU）**：精准抓出视频生成流水线里的 GPU 空等气泡（Bubble Time）。
2. **首尾无损异步重叠（Phase 1）**：将读图解码（IO+CPU）与 H.264 视频编码落盘（CPU+IO）同 GPU 计算重叠，同时受内存水位保护。
3. **可视化泳道图 + Perfetto JSON 导出**：直观看到消除了哪些气泡、还剩哪些气泡留给第二阶段（PCIe 权重搬运）。


### Step 1：检查 T4 GPU 硬件环境并安装依赖


In [ ]:
!nvidia-smi
!lscpu | grep -E 'Model name|CPU\(s\):'
!free -h
!pip install -q diffusers transformers accelerate safetensors psutil pillow matplotlib


### Step 2：部署 Bubbleless v0.1 核心模块到 `/content/bubbleless`


In [ ]:
import pathlib
ROOT = pathlib.Path('/content/bubbleless')
(ROOT / 'bubbleless').mkdir(parents=True, exist_ok=True)

(ROOT / 'bubbleless/__init__.py').write_text('"""Bubbleless v0.1 - Cross-component (IO/CPU/GPU) bubble profiler & scheduler."""\n\n__version__ = "0.1.0"\n', encoding='utf-8')
(ROOT / 'bubbleless/profiler.py').write_text('"""Thread-safe IO / CPU / GPU Timeline Profiler & Perfetto Trace Exporter."""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport dataclasses\nimport json\nimport pathlib\nimport threading\nimport time\nfrom typing import Dict, Iterator, List, Literal, Optional, Tuple\n\nComponentType = Literal["IO", "CPU", "GPU"]\n\nLANE_PID_MAP: Dict[str, Tuple[int, str]] = {\n    "BUBBLE": (0, "0_GPU_Bubble_Diagnosis"),\n    "GPU": (1, "1_GPU_Compute"),\n    "CPU": (2, "2_CPU_PrePostProcess"),\n    "IO": (3, "3_IO_Disk_and_PCIe"),\n}\n\n\ndef _maybe_cuda_sync() -> None:\n    try:\n        import torch\n\n        if torch.cuda.is_available():\n            torch.cuda.synchronize()\n    except ImportError:\n        pass\n\n\n@dataclasses.dataclass\nclass TraceSpan:\n    component: ComponentType\n    name: str\n    task_id: int\n    start_s: float  # Relative to profiler start\n    end_s: float\n    thread_id: int\n    meta: Dict[str, object] = dataclasses.field(default_factory=dict)\n\n    @property\n    def duration_s(self) -> float:\n        return max(0.0, self.end_s - self.start_s)\n\n\ndef merge_intervals(intervals: List[Tuple[float, float]]) -> List[Tuple[float, float]]:\n    """Merges overlapping [start, end] intervals."""\n    if not intervals:\n        return []\n    sorted_iv = sorted(intervals, key=lambda x: x[0])\n    merged = [sorted_iv[0]]\n    for cur_s, cur_e in sorted_iv[1:]:\n        prev_s, prev_e = merged[-1]\n        if cur_s <= prev_e:\n            merged[-1] = (prev_s, max(prev_e, cur_e))\n        else:\n            merged.append((cur_s, cur_e))\n    return merged\n\n\ndef intersect_duration(\n    window: Tuple[float, float], intervals: List[Tuple[float, float]]\n) -> float:\n    """Computes total overlap between `window` and already-merged `intervals`."""\n    w_s, w_e = window\n    total = 0.0\n    for iv_s, iv_e in intervals:\n        if iv_e <= w_s:\n            continue\n        if iv_s >= w_e:\n            break\n        total += min(w_e, iv_e) - max(w_s, iv_s)\n    return total\n\n\nclass TriComponentProfiler:\n    """Records IO, CPU, and GPU spans across threads and diagnoses GPU bubbles."""\n\n    def __init__(self, run_name: str = "experiment") -> None:\n        self.run_name = run_name\n        self._lock = threading.Lock()\n        self._spans: List[TraceSpan] = []\n        self._t0: Optional[float] = None\n        self._t_end: Optional[float] = None\n        self.peak_ram_mb: float = 0.0\n        self.peak_vram_mb: float = 0.0\n\n    def start_session(self) -> None:\n        _maybe_cuda_sync()\n        self._t0 = time.perf_counter()\n        self._spans.clear()\n        self.sample_memory()\n\n    def stop_session(self) -> None:\n        _maybe_cuda_sync()\n        self._t_end = time.perf_counter()\n        self.sample_memory()\n\n    def sample_memory(self) -> Tuple[float, float]:\n        """Samples current RSS RAM (MB) and CUDA max allocated VRAM (MB)."""\n        ram_mb = 0.0\n        vram_mb = 0.0\n        try:\n            import psutil\n\n            ram_mb = psutil.Process().memory_info().rss / (1024 * 1024)\n        except ImportError:\n            import resource\n\n            ram_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024.0\n\n        try:\n            import torch\n\n            if torch.cuda.is_available():\n                vram_mb = torch.cuda.max_memory_allocated() / (1024 * 1024)\n        except ImportError:\n            pass\n\n        with self._lock:\n            self.peak_ram_mb = max(self.peak_ram_mb, ram_mb)\n            self.peak_vram_mb = max(self.peak_vram_mb, vram_mb)\n        return ram_mb, vram_mb\n\n    @contextlib.contextmanager\n    def span(\n        self,\n        component: ComponentType,\n        name: str,\n        task_id: int = -1,\n        cuda_sync: bool = False,\n        **meta: object,\n    ) -> Iterator[None]:\n        """Context manager to record an IO, CPU, or GPU operation."""\n        if self._t0 is None:\n            self.start_session()\n        assert self._t0 is not None\n\n        if cuda_sync or component == "GPU":\n            _maybe_cuda_sync()\n        t_start = time.perf_counter() - self._t0\n        try:\n            yield\n        finally:\n            if cuda_sync or component == "GPU":\n                _maybe_cuda_sync()\n            t_end = time.perf_counter() - self._t0\n            tid = threading.get_ident()\n            with self._lock:\n                self._spans.append(\n                    TraceSpan(\n                        component=component,\n                        name=name,\n                        task_id=task_id,\n                        start_s=t_start,\n                        end_s=t_end,\n                        thread_id=tid,\n                        meta=dict(meta),\n                    )\n                )\n            self.sample_memory()\n\n    @property\n    def total_wall_s(self) -> float:\n        if self._t0 is None:\n            return 0.0\n        end = self._t_end if self._t_end is not None else time.perf_counter()\n        return max(0.0, end - self._t0)\n\n    def _compute_gpu_bubbles(\n        self,\n    ) -> Tuple[List[Tuple[float, float, str]], Dict[str, float]]:\n        """Finds all intervals where GPU is idle and attributes who caused the wait."""\n        total_s = self.total_wall_s\n        if total_s <= 0:\n            return [], {}\n\n        with self._lock:\n            spans_copy = list(self._spans)\n\n        gpu_tids = {s.thread_id for s in spans_copy if s.component == "GPU"}\n        gpu_iv = merge_intervals(\n            [(s.start_s, s.end_s) for s in spans_copy if s.component == "GPU"]\n        )\n        main_io_iv = merge_intervals(\n            [\n                (s.start_s, s.end_s)\n                for s in spans_copy\n                if s.component == "IO" and s.thread_id in gpu_tids\n            ]\n        )\n        main_cpu_iv = merge_intervals(\n            [\n                (s.start_s, s.end_s)\n                for s in spans_copy\n                if s.component == "CPU" and s.thread_id in gpu_tids\n            ]\n        )\n        cpu_iv = merge_intervals(\n            [(s.start_s, s.end_s) for s in spans_copy if s.component == "CPU"]\n        )\n        io_iv = merge_intervals(\n            [(s.start_s, s.end_s) for s in spans_copy if s.component == "IO"]\n        )\n\n        # Invert gpu_iv within [0, total_s] to get GPU idle gaps (bubbles)\n        bubbles: List[Tuple[float, float, str]] = []\n        cursor = 0.0\n        for g_s, g_e in gpu_iv:\n            if g_s > cursor:\n                bubbles.append((cursor, g_s, ""))\n            cursor = max(cursor, g_e)\n        if cursor < total_s:\n            bubbles.append((cursor, total_s, ""))\n\n        attributed_bubbles: List[Tuple[float, float, str]] = []\n        wait_io_s = 0.0\n        wait_cpu_s = 0.0\n        wait_other_s = 0.0\n\n        for b_s, b_e, _ in bubbles:\n            dur = b_e - b_s\n            if dur <= 1e-4:\n                continue\n            # 1. Causal priority: check how much of this GPU idle window the GPU thread itself spent in IO / CPU\n            m_io = intersect_duration((b_s, b_e), main_io_iv)\n            m_cpu = intersect_duration((b_s, b_e), main_cpu_iv)\n            if (m_io + m_cpu) > 0.002:\n                scale = min(1.0, dur / (m_io + m_cpu))\n                io_part = m_io * scale\n                cpu_part = m_cpu * scale\n                rem = max(0.0, dur - io_part - cpu_part)\n                wait_io_s += io_part\n                wait_cpu_s += cpu_part\n                wait_other_s += rem\n                reason = "GPU Waiting on IO" if io_part >= cpu_part else "GPU Waiting on CPU"\n            else:\n                # 2. GPU thread was blocked waiting on async feeder/exporter queue\n                io_overlap = intersect_duration((b_s, b_e), io_iv)\n                cpu_overlap = intersect_duration((b_s, b_e), cpu_iv)\n                if (io_overlap + cpu_overlap) > 0.002:\n                    scale = min(1.0, dur / (io_overlap + cpu_overlap))\n                    io_part = io_overlap * scale\n                    cpu_part = cpu_overlap * scale\n                    rem = max(0.0, dur - io_part - cpu_part)\n                    wait_io_s += io_part\n                    wait_cpu_s += cpu_part\n                    wait_other_s += rem\n                    reason = "GPU Waiting on IO" if io_part >= cpu_part else "GPU Waiting on CPU"\n                else:\n                    reason = "GPU Unattributed Gap"\n                    wait_other_s += dur\n            attributed_bubbles.append((b_s, b_e, reason))\n\n        gpu_busy_s = sum(e - s for s, e in gpu_iv)\n        cpu_busy_s = sum(e - s for s, e in cpu_iv)\n        io_busy_s = sum(e - s for s, e in io_iv)\n        bubble_total_s = max(0.0, total_s - gpu_busy_s)\n\n        summary = {\n            "total_wall_s": total_s,\n            "gpu_busy_s": gpu_busy_s,\n            "cpu_busy_s": cpu_busy_s,\n            "io_busy_s": io_busy_s,\n            "gpu_bubble_s": bubble_total_s,\n            "gpu_bubble_pct": (bubble_total_s / total_s * 100.0) if total_s > 0 else 0.0,\n            "gpu_util_pct": (gpu_busy_s / total_s * 100.0) if total_s > 0 else 0.0,\n            "bubble_wait_io_s": wait_io_s,\n            "bubble_wait_cpu_s": wait_cpu_s,\n            "bubble_wait_other_s": wait_other_s,\n            "peak_ram_mb": self.peak_ram_mb,\n            "peak_vram_mb": self.peak_vram_mb,\n        }\n        return attributed_bubbles, summary\n\n    def summary(self) -> Dict[str, float]:\n        _, stats = self._compute_gpu_bubbles()\n        return stats\n\n    def format_report(self) -> str:\n        s = self.summary()\n        lines = [\n            f"=== Bubbleless Profiler Report [{self.run_name}] ===",\n            f"  End-to-End Wall Time : {s[\'total_wall_s\']:.2f} s",\n            f"  GPU Active Compute   : {s[\'gpu_busy_s\']:.2f} s ({s[\'gpu_util_pct\']:.1f}%)",\n            f"  GPU Bubble (Waiting) : {s[\'gpu_bubble_s\']:.2f} s ({s[\'gpu_bubble_pct\']:.1f}%)",\n            f"    ├─ Waiting on IO   : {s[\'bubble_wait_io_s\']:.2f} s",\n            f"    ├─ Waiting on CPU  : {s[\'bubble_wait_cpu_s\']:.2f} s",\n            f"    └─ Scheduling Gap  : {s[\'bubble_wait_other_s\']:.2f} s",\n            f"  CPU Active Time      : {s[\'cpu_busy_s\']:.2f} s",\n            f"  IO Active Time       : {s[\'io_busy_s\']:.2f} s",\n            f"  Peak Host RAM / VRAM : {s[\'peak_ram_mb\']:.0f} MB / {s[\'peak_vram_mb\']:.0f} MB",\n        ]\n        return "\\n".join(lines)\n\n    def export_perfetto_json(self, output_path: str | pathlib.Path) -> pathlib.Path:\n        """Exports Chrome Tracing / Perfetto UI compatible JSON with IO/CPU/GPU swimlanes."""\n        path = pathlib.Path(output_path)\n        path.parent.mkdir(parents=True, exist_ok=True)\n        bubbles, _ = self._compute_gpu_bubbles()\n\n        events: List[Dict[str, object]] = []\n        # Process sort labels so Perfetto orders lanes nicely: Bubble -> GPU -> CPU -> IO\n        for comp_key, (pid, proc_name) in LANE_PID_MAP.items():\n            events.append(\n                {\n                    "name": "process_name",\n                    "ph": "M",\n                    "pid": pid,\n                    "tid": 0,\n                    "args": {"name": proc_name},\n                }\n            )\n            events.append(\n                {\n                    "name": "process_sort_index",\n                    "ph": "M",\n                    "pid": pid,\n                    "tid": 0,\n                    "args": {"sort_index": pid},\n                }\n            )\n\n        with self._lock:\n            spans_copy = list(self._spans)\n\n        for sp in spans_copy:\n            pid, _ = LANE_PID_MAP[sp.component]\n            label = f"[T{sp.task_id}] {sp.name}" if sp.task_id >= 0 else sp.name\n            events.append(\n                {\n                    "name": label,\n                    "cat": sp.component,\n                    "ph": "X",\n                    "ts": round(sp.start_s * 1_000_000, 2),\n                    "dur": round(sp.duration_s * 1_000_000, 2),\n                    "pid": pid,\n                    "tid": 1,\n                    "args": {"task_id": sp.task_id, **sp.meta},\n                }\n            )\n\n        # Add explicit GPU Bubble spans on lane 0 (only bubbles >= 5ms to avoid clutter)\n        bubble_pid, _ = LANE_PID_MAP["BUBBLE"]\n        for b_s, b_e, reason in bubbles:\n            if (b_e - b_s) >= 0.005:\n                events.append(\n                    {\n                        "name": f"BUBBLE: {reason}",\n                        "cat": "BUBBLE",\n                        "ph": "X",\n                        "ts": round(b_s * 1_000_000, 2),\n                        "dur": round((b_e - b_s) * 1_000_000, 2),\n                        "pid": bubble_pid,\n                        "tid": 1,\n                        "args": {"reason": reason, "duration_ms": round((b_e - b_s) * 1000, 1)},\n                    }\n                )\n\n        payload = {"traceEvents": events, "displayTimeUnit": "ms"}\n        path.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n        return path\n', encoding='utf-8')
(ROOT / 'bubbleless/scheduler.py').write_text('"""Policy & Mechanism separated pipeline scheduler for Phase 1 (v0.1)."""\n\nfrom __future__ import annotations\n\nimport dataclasses\nimport queue\nimport threading\nimport time\nfrom typing import Any, Dict, Generic, List, Literal, Protocol, TypeVar\n\nfrom .profiler import TriComponentProfiler\n\nT_In = TypeVar("T_In")\nT_Prep = TypeVar("T_Prep")\nT_RawOut = TypeVar("T_RawOut")\nT_Final = TypeVar("T_Final")\n\n\n@dataclasses.dataclass\nclass SchedulePolicy:\n    """Policy layer: resource watermarks and bounded concurrency rules."""\n\n    mode: Literal["serial", "async_overlap"] = "async_overlap"\n    max_prefetch_queue: int = 2\n    max_export_queue: int = 2\n    min_free_ram_mb: float = 1500.0  # Safety guard for Colab 12.7GB RAM\n\n    def allow_prefetch(self) -> bool:\n        """Checks whether host RAM has enough headroom to prefetch the next task."""\n        try:\n            import psutil\n\n            avail_mb = psutil.virtual_memory().available / (1024 * 1024)\n            return avail_mb >= self.min_free_ram_mb\n        except ImportError:\n            return True\n\n\nclass GenerativeWorkload(Protocol[T_In, T_Prep, T_RawOut, T_Final]):\n    """Interface for any multi-stage generative workflow (Image/Video generation)."""\n\n    def preprocess_io_cpu(self, task_id: int, item: T_In, prof: TriComponentProfiler) -> T_Prep:\n        """Stage 1: Disk read + CPU image/video decode & resize."""\n        ...\n\n    def infer_gpu(self, task_id: int, prep: T_Prep, prof: TriComponentProfiler) -> T_RawOut:\n        """Stage 2: PCIe model/tensor transfer + GPU denoising & VAE decode (returns CPU-backed frames)."""\n        ...\n\n    def export_cpu_io(self, task_id: int, raw: T_RawOut, prof: TriComponentProfiler) -> T_Final:\n        """Stage 3: CPU video/image encoding + Disk write."""\n        ...\n\n\ndef _put_interruptible(q: "queue.Queue[Any]", item: Any, stop_evt: threading.Event) -> None:\n    while not stop_evt.is_set():\n        try:\n            q.put(item, timeout=0.1)\n            return\n        except queue.Full:\n            continue\n    raise RuntimeError("Pipeline aborted due to upstream/downstream error.")\n\n\ndef _get_interruptible(q: "queue.Queue[Any]", stop_evt: threading.Event) -> Any:\n    while not stop_evt.is_set():\n        try:\n            return q.get(timeout=0.1)\n        except queue.Empty:\n            continue\n    raise RuntimeError("Pipeline aborted due to upstream/downstream error.")\n\n\nclass PipelineExecutor(Generic[T_In, T_Prep, T_RawOut, T_Final]):\n    """Mechanism layer: executes tasks according to SchedulePolicy."""\n\n    def __init__(\n        self,\n        workload: GenerativeWorkload[T_In, T_Prep, T_RawOut, T_Final],\n        policy: SchedulePolicy,\n    ) -> None:\n        self.workload = workload\n        self.policy = policy\n\n    def run_batch(\n        self, tasks: List[T_In], profiler: TriComponentProfiler\n    ) -> List[T_Final]:\n        if self.policy.mode == "serial":\n            return self._run_serial(tasks, profiler)\n        return self._run_async_overlap(tasks, profiler)\n\n    def _run_serial(\n        self, tasks: List[T_In], profiler: TriComponentProfiler\n    ) -> List[T_Final]:\n        results: List[T_Final] = []\n        profiler.start_session()\n        try:\n            for task_id, item in enumerate(tasks):\n                prep = self.workload.preprocess_io_cpu(task_id, item, profiler)\n                raw = self.workload.infer_gpu(task_id, prep, profiler)\n                final = self.workload.export_cpu_io(task_id, raw, profiler)\n                results.append(final)\n        finally:\n            profiler.stop_session()\n        return results\n\n    def _run_async_overlap(\n        self, tasks: List[T_In], profiler: TriComponentProfiler\n    ) -> List[T_Final]:\n        """Double-buffered async feeder (IO+CPU) + main thread GPU + async exporter (CPU+IO)."""\n        n_tasks = len(tasks)\n        prefetch_q: "queue.Queue[Any]" = queue.Queue(maxsize=self.policy.max_prefetch_queue)\n        export_q: "queue.Queue[Any]" = queue.Queue(maxsize=self.policy.max_export_queue)\n        stop_evt = threading.Event()\n        errors: List[BaseException] = []\n        results_map: Dict[int, T_Final] = {}\n        SENTINEL = object()\n\n        def feeder_loop() -> None:\n            try:\n                for task_id, item in enumerate(tasks):\n                    while not stop_evt.is_set() and not self.policy.allow_prefetch():\n                        # Backpressure if RAM drops below watermark ( only block if queue already has >=1 item )\n                        if prefetch_q.empty():\n                            break\n                        time.sleep(0.05)\n                    prep = self.workload.preprocess_io_cpu(task_id, item, profiler)\n                    _put_interruptible(prefetch_q, (task_id, prep), stop_evt)\n                _put_interruptible(prefetch_q, SENTINEL, stop_evt)\n            except BaseException as exc:  # noqa: BLE001\n                errors.append(exc)\n                stop_evt.set()\n\n        def exporter_loop() -> None:\n            try:\n                while not stop_evt.is_set():\n                    pkt = _get_interruptible(export_q, stop_evt)\n                    if pkt is SENTINEL:\n                        break\n                    task_id, raw = pkt\n                    final = self.workload.export_cpu_io(task_id, raw, profiler)\n                    results_map[task_id] = final\n            except BaseException as exc:  # noqa: BLE001\n                errors.append(exc)\n                stop_evt.set()\n\n        feeder_t = threading.Thread(target=feeder_loop, name="Bubbleless-Feeder", daemon=True)\n        exporter_t = threading.Thread(target=exporter_loop, name="Bubbleless-Exporter", daemon=True)\n\n        profiler.start_session()\n        try:\n            feeder_t.start()\n            exporter_t.start()\n\n            # GPU execution strictly stays on main thread (CUDA stream & accelerate hook safe)\n            for _ in range(n_tasks):\n                pkt = _get_interruptible(prefetch_q, stop_evt)\n                if pkt is SENTINEL:\n                    break\n                task_id, prep = pkt\n                raw = self.workload.infer_gpu(task_id, prep, profiler)\n                _put_interruptible(export_q, (task_id, raw), stop_evt)\n\n            _put_interruptible(export_q, SENTINEL, stop_evt)\n            feeder_t.join()\n            exporter_t.join()\n        except BaseException as exc:\n            stop_evt.set()\n            raise exc\n        finally:\n            profiler.stop_session()\n\n        if errors:\n            raise RuntimeError(f"Worker thread failed: {errors[0]}") from errors[0]\n\n        return [results_map[i] for i in range(n_tasks)]\n', encoding='utf-8')
(ROOT / 'bubbleless/workload_t4.py').write_text('"""Colab T4 Video Generation Workload (Diffusers AnimateDiff fp16 + Synthetic T4 Dry-run)."""\n\nfrom __future__ import annotations\n\nimport dataclasses\nimport hashlib\nimport io\nimport os\nimport pathlib\nimport subprocess\nimport time\nfrom typing import Any, Callable, List, Optional, Tuple\n\nimport numpy as np\nfrom PIL import Image, ImageDraw, ImageFilter\n\nfrom .profiler import TriComponentProfiler\n\n\n@dataclasses.dataclass\nclass VideoTaskSpec:\n    task_id: int\n    prompt: str\n    seed: int\n    source_frames_dir: pathlib.Path\n    output_dir: pathlib.Path\n    strength: float = 0.65\n    num_inference_steps: int = 12\n\n\n@dataclasses.dataclass\nclass PreparedVideoTask:\n    spec: VideoTaskSpec\n    frames_pil: List[Image.Image]\n\n\n@dataclasses.dataclass\nclass RawGeneratedVideo:\n    spec: VideoTaskSpec\n    frames_np: np.ndarray  # uint8 shape (F, H, W, 3) on CPU RAM\n\n\n@dataclasses.dataclass\nclass ExportedVideoArtifact:\n    task_id: int\n    mp4_path: pathlib.Path\n    preview_png_path: pathlib.Path\n    frame_sha256: str\n\n\ndef generate_test_source_clips(\n    root_dir: str | pathlib.Path,\n    num_tasks: int = 6,\n    num_frames: int = 16,\n    src_resolution: int = 1280,\n) -> List[VideoTaskSpec]:\n    """Creates deterministic high-res reference frame sequences on local disk (/content/...)."""\n    base = pathlib.Path(root_dir)\n    src_root = base / "inputs"\n    out_root = base / "outputs"\n    src_root.mkdir(parents=True, exist_ok=True)\n    out_root.mkdir(parents=True, exist_ok=True)\n\n    prompts = [\n        "cyberpunk neon alley in rain, cinematic lighting, anime film style, highly detailed",\n        "golden hour ocean waves crashing on cliffs, studio ghibli painting style, vibrant",\n        "futuristic astronaut walking through bioluminescent forest, 4k concept art",\n        "steampunk clockwork train crossing mountain bridge in snow, dramatic clouds",\n        "ancient japanese temple under falling cherry blossoms at dusk, warm lanterns",\n        "watercolor sailboat drifting on misty alpine lake at sunrise, masterpiece",\n        "retro synthwave sports car driving toward glowing sunset grid, 80s aesthetic",\n        "magical library with floating glowing books and warm fireplace, fantasy art",\n    ]\n\n    specs: List[VideoTaskSpec] = []\n    for tid in range(num_tasks):\n        clip_dir = src_root / f"clip_{tid:02d}"\n        clip_dir.mkdir(parents=True, exist_ok=True)\n        # Only generate PNGs if not cached already\n        first_frame = clip_dir / "frame_00.png"\n        if not first_frame.exists():\n            rng = np.random.default_rng(1000 + tid)\n            bg_noise = rng.integers(15, 65, size=(src_resolution, src_resolution, 3), dtype=np.uint8)\n            for f_idx in range(num_frames):\n                canvas = Image.fromarray(bg_noise.copy(), mode="RGB")\n                draw = ImageDraw.Draw(canvas)\n                # Moving geometric subject across frames so video2video has coherent motion\n                cx = int(src_resolution * (0.2 + 0.6 * (f_idx / max(1, num_frames - 1))))\n                cy = int(src_resolution * (0.5 + 0.15 * np.sin(f_idx * 0.5 + tid)))\n                r = src_resolution // 6\n                color = (\n                    int((80 + tid * 35) % 255),\n                    int((140 + f_idx * 7) % 255),\n                    220,\n                )\n                draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=color, outline=(255, 255, 255), width=6)\n                canvas.save(clip_dir / f"frame_{f_idx:02d}.png", compress_level=3)\n\n        specs.append(\n            VideoTaskSpec(\n                task_id=tid,\n                prompt=prompts[tid % len(prompts)],\n                seed=42 + tid,\n                source_frames_dir=clip_dir,\n                output_dir=out_root,\n            )\n        )\n    return specs\n\n\ndef _encode_mp4_bytes(frames_np: np.ndarray, fps: int = 8) -> bytes:\n    """Encodes (F, H, W, 3) uint8 frames into H.264 MP4 bytes on CPU via ffmpeg."""\n    f_count, height, width, _ = frames_np.shape\n    cmd = [\n        "ffmpeg",\n        "-y",\n        "-f",\n        "rawvideo",\n        "-vcodec",\n        "rawvideo",\n        "-s",\n        f"{width}x{height}",\n        "-pix_fmt",\n        "rgb24",\n        "-r",\n        str(fps),\n        "-i",\n        "-",\n        "-an",\n        "-vcodec",\n        "libx264",\n        "-preset",\n        "slow",  # Realistic studio archival encode quality on CPU\n        "-crf",\n        "18",\n        "-pix_fmt",\n        "yuv420p",\n        "-movflags",\n        "frag_keyframe+empty_moov",\n        "-f",\n        "mp4",\n        "pipe:1",\n    ]\n    proc = subprocess.run(\n        cmd,\n        input=frames_np.tobytes(),\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        check=True,\n    )\n    return proc.stdout\n\n\ndef _build_contact_sheet_png_bytes(frames_np: np.ndarray) -> bytes:\n    """Builds a 4x4 storyboard contact sheet PNG in memory (CPU heavy)."""\n    n_frames, h, w, _ = frames_np.shape\n    cols = 4\n    rows = (n_frames + cols - 1) // cols\n    sheet = Image.new("RGB", (cols * w, rows * h))\n    for idx in range(n_frames):\n        im = Image.fromarray(frames_np[idx], mode="RGB")\n        r, c = divmod(idx, cols)\n        sheet.paste(im, (c * w, r * h))\n    buf = io.BytesIO()\n    sheet.save(buf, format="PNG", compress_level=6)\n    return buf.getvalue()\n\n\ndef _write_and_fsync(path: pathlib.Path, data: bytes) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with open(path, "wb") as f:\n        f.write(data)\n        f.flush()\n        os.fsync(f.fileno())\n\n\nclass DiffusersT4VideoWorkload:\n    """Real CUDA Video-to-Video workflow for Colab T4 (15GB VRAM, fp16, model offload)."""\n\n    def __init__(\n        self,\n        base_model_id: str = "Lykon/dreamshaper-8",\n        motion_adapter_id: str = "guoyww/animatediff-motion-adapter-v1-5-2",\n        target_size: int = 512,\n        enable_offload: bool = True,\n    ) -> None:\n        import torch\n        from diffusers import AnimateDiffVideoToVideoPipeline, DDIMScheduler, MotionAdapter\n\n        if not torch.cuda.is_available():\n            raise RuntimeError(\n                "CUDA GPU not detected! Switch Google Colab Runtime -> Change runtime type -> T4 GPU."\n            )\n\n        self.target_size = target_size\n        self._active_prof: Optional[TriComponentProfiler] = None\n        self._active_task_id: int = -1\n\n        print(f"[Bubbleless] Loading MotionAdapter ({motion_adapter_id}) in fp16...")\n        adapter = MotionAdapter.from_pretrained(motion_adapter_id, torch_dtype=torch.float16)\n        print(f"[Bubbleless] Loading Base Pipeline ({base_model_id}) in fp16...")\n        self.pipe = AnimateDiffVideoToVideoPipeline.from_pretrained(\n            base_model_id,\n            motion_adapter=adapter,\n            torch_dtype=torch.float16,\n        )\n        self.pipe.scheduler = DDIMScheduler.from_config(\n            self.pipe.scheduler.config,\n            clip_sample=False,\n            timestep_spacing="linspace",\n            beta_schedule="linear",\n            steps_offset=1,\n        )\n        self.pipe.vae.enable_slicing()\n\n        if enable_offload:\n            # Simulates tight VRAM studio environment where models swap CPU <-> GPU over PCIe\n            self.pipe.enable_model_cpu_offload()\n        else:\n            self.pipe.to("cuda")\n\n        self._instrument_pipeline_submodules()\n\n    def _instrument_pipeline_submodules(self) -> None:\n        """Hooks text_encoder, unet, and vae so PCIe Offload IO and GPU Compute never overlap."""\n\n        def _wrap_compute_module(mod: Any, label: str) -> None:\n            # If accelerate wrapped forward, _old_forward is the inner compute body AFTER pre_forward IO!\n            target_attr = "_old_forward" if hasattr(mod, "_old_forward") else "forward"\n            orig_fn = getattr(mod, target_attr)\n\n            def wrapped_compute(*args: Any, **kwargs: Any) -> Any:\n                prof = self._active_prof\n                if prof is None:\n                    return orig_fn(*args, **kwargs)\n                with prof.span(\n                    "GPU",\n                    f"GPU: {label}",\n                    task_id=self._active_task_id,\n                    cuda_sync=True,\n                ):\n                    return orig_fn(*args, **kwargs)\n\n            setattr(mod, target_attr, wrapped_compute)\n\n        def _wrap_offload_hook(mod: Any, name: str) -> None:\n            hf_hook = getattr(mod, "_hf_hook", None)\n            if hf_hook is not None and hasattr(hf_hook, "pre_forward"):\n                orig_pre = hf_hook.pre_forward\n\n                def wrapped_pre(module: Any, *args: Any, **kwargs: Any) -> Any:\n                    prof = self._active_prof\n                    if prof is None:\n                        return orig_pre(module, *args, **kwargs)\n                    was_on_cpu = False\n                    first_param = next(module.parameters(), None)\n                    if first_param is not None and first_param.device.type == "cpu":\n                        was_on_cpu = True\n\n                    if was_on_cpu:\n                        with prof.span(\n                            "IO",\n                            f"PCIe Offload: {name} -> GPU",\n                            task_id=self._active_task_id,\n                            cuda_sync=True,\n                        ):\n                            return orig_pre(module, *args, **kwargs)\n                    return orig_pre(module, *args, **kwargs)\n\n                hf_hook.pre_forward = wrapped_pre\n            elif getattr(mod, "_forward_pre_hooks", None):\n                # Fallback if diffusers used PyTorch forward_pre_hooks for CPU offload\n                existing_hooks = list(mod._forward_pre_hooks.items())\n                mod._forward_pre_hooks.clear()\n\n                def timed_pre_hook(module: Any, args: Any, kwargs: Any) -> Any:\n                    prof = self._active_prof\n                    was_on_cpu = False\n                    first_param = next(module.parameters(), None)\n                    if first_param is not None and first_param.device.type == "cpu":\n                        was_on_cpu = True\n\n                    def _run_all() -> Any:\n                        cur_args, cur_kwargs = args, kwargs\n                        for _, hk in existing_hooks:\n                            res = hk(module, cur_args, cur_kwargs)\n                            if res is not None:\n                                cur_args, cur_kwargs = res\n                        return cur_args, cur_kwargs\n\n                    if prof is not None and was_on_cpu:\n                        with prof.span(\n                            "IO",\n                            f"PCIe Offload: {name} -> GPU",\n                            task_id=self._active_task_id,\n                            cuda_sync=True,\n                        ):\n                            return _run_all()\n                    return _run_all()\n\n                mod.register_forward_pre_hook(timed_pre_hook, with_kwargs=True)\n\n        for comp_name in ("text_encoder", "unet", "vae"):\n            mod = getattr(self.pipe, comp_name, None)\n            if mod is None:\n                continue\n            _wrap_offload_hook(mod, comp_name)\n            if comp_name == "vae":\n                if hasattr(mod, "encoder"):\n                    _wrap_offload_hook(mod.encoder, "vae.encoder")\n                    _wrap_compute_module(mod.encoder, "vae.encode")\n                if hasattr(mod, "decoder"):\n                    _wrap_offload_hook(mod.decoder, "vae.decoder")\n                    _wrap_compute_module(mod.decoder, "vae.decode")\n            else:\n                _wrap_compute_module(mod, comp_name)\n\n    def warmup(self, sample_spec: VideoTaskSpec) -> None:\n        """Runs a tiny 2-step warmup so one-time CUDA cublas/cudnn init doesn\'t skew benchmarks."""\n        import torch\n\n        print("[Bubbleless] Running quick CUDA kernel warmup...")\n        dummy_prof = TriComponentProfiler("warmup")\n        prep = self.preprocess_io_cpu(0, sample_spec, dummy_prof)\n        gen = torch.Generator(device="cpu").manual_seed(sample_spec.seed)\n        _ = self.pipe(\n            video=prep.frames_pil,\n            prompt=sample_spec.prompt,\n            num_inference_steps=2,\n            strength=0.6,\n            generator=gen,\n            output_type="np",\n        )\n        torch.cuda.synchronize()\n        torch.cuda.reset_peak_memory_stats()\n\n    def preprocess_io_cpu(\n        self, task_id: int, item: VideoTaskSpec, prof: TriComponentProfiler\n    ) -> PreparedVideoTask:\n        frame_files = sorted(item.source_frames_dir.glob("frame_*.png"))\n        raw_blobs: List[bytes] = []\n        with prof.span("IO", "Disk Read: 16 Source PNGs", task_id=task_id):\n            for fp in frame_files:\n                raw_blobs.append(fp.read_bytes())\n\n        frames_pil: List[Image.Image] = []\n        with prof.span("CPU", "CPU Preprocess: PNG Decode + Lanczos + Filter", task_id=task_id):\n            for blob in raw_blobs:\n                im = Image.open(io.BytesIO(blob)).convert("RGB")\n                im = im.resize((self.target_size, self.target_size), Image.Resampling.LANCZOS)\n                im = im.filter(ImageFilter.UnsharpMask(radius=1.5, percent=120, threshold=3))\n                frames_pil.append(im)\n\n        return PreparedVideoTask(spec=item, frames_pil=frames_pil)\n\n    def infer_gpu(\n        self, task_id: int, prep: PreparedVideoTask, prof: TriComponentProfiler\n    ) -> RawGeneratedVideo:\n        import torch\n\n        self._active_prof = prof\n        self._active_task_id = task_id\n        try:\n            gen = torch.Generator(device="cpu").manual_seed(prep.spec.seed)\n            out = self.pipe(\n                video=prep.frames_pil,\n                prompt=prep.spec.prompt,\n                negative_prompt="blurry, low quality, distorted, watermark",\n                strength=prep.spec.strength,\n                num_inference_steps=prep.spec.num_inference_steps,\n                guidance_scale=7.5,\n                generator=gen,\n                output_type="np",\n            )\n            # out.frames[0] is float32 [0,1] numpy array of shape (F, H, W, 3) on CPU\n            frames_float = out.frames[0]\n            frames_u8 = np.clip(np.round(frames_float * 255.0), 0, 255).astype(np.uint8)\n            return RawGeneratedVideo(spec=prep.spec, frames_np=frames_u8)\n        finally:\n            self._active_prof = None\n            self._active_task_id = -1\n\n    def export_cpu_io(\n        self, task_id: int, raw: RawGeneratedVideo, prof: TriComponentProfiler\n    ) -> ExportedVideoArtifact:\n        with prof.span("CPU", "CPU Encode: H.264 MP4 + Storyboard Sheet", task_id=task_id):\n            mp4_bytes = _encode_mp4_bytes(raw.frames_np, fps=8)\n            png_bytes = _build_contact_sheet_png_bytes(raw.frames_np)\n            digest = hashlib.sha256(raw.frames_np.tobytes()).hexdigest()[:16]\n\n        mp4_path = raw.spec.output_dir / f"video_{task_id:02d}.mp4"\n        png_path = raw.spec.output_dir / f"storyboard_{task_id:02d}.png"\n        with prof.span("IO", "Disk Write: MP4 + Storyboard (fsync)", task_id=task_id):\n            _write_and_fsync(mp4_path, mp4_bytes)\n            _write_and_fsync(png_path, png_bytes)\n\n        return ExportedVideoArtifact(\n            task_id=task_id,\n            mp4_path=mp4_path,\n            preview_png_path=png_path,\n            frame_sha256=digest,\n        )\n\n\nclass SyntheticT4Workload:\n    """CPU-runnable synthetic workload calibrated to Colab T4 proportions for quick dry-runs."""\n\n    def __init__(self, target_size: int = 512, scale_time: float = 0.25) -> None:\n        self.target_size = target_size\n        self.scale = scale_time\n\n    def preprocess_io_cpu(\n        self, task_id: int, item: VideoTaskSpec, prof: TriComponentProfiler\n    ) -> PreparedVideoTask:\n        frame_files = sorted(item.source_frames_dir.glob("frame_*.png"))\n        raw_blobs: List[bytes] = []\n        with prof.span("IO", "Disk Read: 16 Source PNGs", task_id=task_id):\n            for fp in frame_files:\n                raw_blobs.append(fp.read_bytes())\n            time.sleep(0.25 * self.scale)\n\n        frames_pil: List[Image.Image] = []\n        with prof.span("CPU", "CPU Preprocess: PNG Decode + Lanczos + Filter", task_id=task_id):\n            for blob in raw_blobs:\n                im = Image.open(io.BytesIO(blob)).convert("RGB")\n                im = im.resize((self.target_size, self.target_size), Image.Resampling.LANCZOS)\n                im = im.filter(ImageFilter.UnsharpMask(radius=1.5, percent=120, threshold=3))\n                frames_pil.append(im)\n\n        return PreparedVideoTask(spec=item, frames_pil=frames_pil)\n\n    def infer_gpu(\n        self, task_id: int, prep: PreparedVideoTask, prof: TriComponentProfiler\n    ) -> RawGeneratedVideo:\n        # Simulates: PCIe load TextEncoder -> GPU Text -> PCIe load UNet -> GPU Denoise -> PCIe load VAE -> GPU VAE\n        with prof.span("IO", "PCIe Offload: text_encoder -> GPU", task_id=task_id):\n            time.sleep(0.20 * self.scale)\n        with prof.span("GPU", "GPU: text_encoder", task_id=task_id):\n            time.sleep(0.12 * self.scale)\n\n        with prof.span("IO", "PCIe Offload: vae -> GPU", task_id=task_id):\n            time.sleep(0.18 * self.scale)\n        with prof.span("GPU", "GPU: vae.encode", task_id=task_id):\n            time.sleep(0.35 * self.scale)\n\n        with prof.span("IO", "PCIe Offload: unet -> GPU", task_id=task_id):\n            time.sleep(0.65 * self.scale)\n        for _ in range(4):\n            with prof.span("GPU", "GPU: unet", task_id=task_id):\n                time.sleep(0.85 * self.scale)\n\n        with prof.span("IO", "PCIe Offload: vae -> GPU", task_id=task_id):\n            time.sleep(0.18 * self.scale)\n        with prof.span("GPU", "GPU: vae.decode", task_id=task_id):\n            time.sleep(0.45 * self.scale)\n\n        # Deterministic output derived from preprocessed frames + seed\n        arr = np.stack([np.asarray(im, dtype=np.uint8) for im in prep.frames_pil], axis=0)\n        arr = ((arr.astype(np.uint16) + (prep.spec.seed % 37)) % 256).astype(np.uint8)\n        return RawGeneratedVideo(spec=prep.spec, frames_np=arr)\n\n    def export_cpu_io(\n        self, task_id: int, raw: RawGeneratedVideo, prof: TriComponentProfiler\n    ) -> ExportedVideoArtifact:\n        with prof.span("CPU", "CPU Encode: H.264 MP4 + Storyboard Sheet", task_id=task_id):\n            mp4_bytes = _encode_mp4_bytes(raw.frames_np, fps=8)\n            png_bytes = _build_contact_sheet_png_bytes(raw.frames_np)\n            digest = hashlib.sha256(raw.frames_np.tobytes()).hexdigest()[:16]\n\n        mp4_path = raw.spec.output_dir / f"video_{task_id:02d}.mp4"\n        png_path = raw.spec.output_dir / f"storyboard_{task_id:02d}.png"\n        with prof.span("IO", "Disk Write: MP4 + Storyboard (fsync)", task_id=task_id):\n            _write_and_fsync(mp4_path, mp4_bytes)\n            _write_and_fsync(png_path, png_bytes)\n\n        return ExportedVideoArtifact(\n            task_id=task_id,\n            mp4_path=mp4_path,\n            preview_png_path=png_path,\n            frame_sha256=digest,\n        )\n', encoding='utf-8')
(ROOT / 'bubbleless/viz.py').write_text('"""Swimlane Timeline Visualizer (renders Serial vs Overlap Gantt chart PNG)."""\n\nfrom __future__ import annotations\n\nimport json\nimport pathlib\nfrom typing import Dict, List, Tuple\n\n\ndef plot_ab_swimlanes(\n    serial_trace_json: str | pathlib.Path,\n    overlap_trace_json: str | pathlib.Path,\n    output_png: str | pathlib.Path,\n) -> pathlib.Path:\n    """Renders a 2-panel horizontal swimlane comparison chart from Perfetto JSON files."""\n    import matplotlib.patches as mpatches\n    import matplotlib.pyplot as plt\n\n    traces = [\n        ("Baseline: Serial Execution", pathlib.Path(serial_trace_json)),\n        ("Phase 1 (v0.1): Async IO/CPU Overlap", pathlib.Path(overlap_trace_json)),\n    ]\n\n    lane_order = ["BUBBLE", "GPU", "CPU", "IO"]\n    lane_labels = {\n        "BUBBLE": "GPU Bubble\\n(Idle Wait)",\n        "GPU": "GPU\\nCompute",\n        "CPU": "CPU\\nPre/Post",\n        "IO": "IO\\nDisk & PCIe",\n    }\n    colors = {\n        "BUBBLE": "#ef4444",  # Red highlight for wasted GPU bubbles\n        "GPU": "#10b981",     # Emerald green for active GPU compute\n        "CPU": "#3b82f6",     # Blue for CPU preprocess & H264 encode\n        "IO": "#f59e0b",      # Amber for Disk & PCIe offload IO\n    }\n\n    parsed_panels: List[Tuple[str, List[Dict[str, object]], float]] = []\n    global_max_s = 0.1\n    for title, p in traces:\n        data = json.loads(p.read_text(encoding="utf-8"))\n        evts = [e for e in data.get("traceEvents", []) if e.get("ph") == "X"]\n        max_end_s = max(\n            ((float(e["ts"]) + float(e["dur"])) / 1_000_000.0 for e in evts),\n            default=0.1,\n        )\n        global_max_s = max(global_max_s, max_end_s)\n        parsed_panels.append((title, evts, max_end_s))\n\n    fig, axes = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)\n    fig.patch.set_facecolor("#0f172a")\n\n    for ax, (title, evts, panel_end_s) in zip(axes, parsed_panels):\n        ax.set_facecolor("#1e293b")\n        for y_idx, cat in enumerate(lane_order):\n            cat_evts = [e for e in evts if e.get("cat") == cat]\n            bars = [\n                (float(e["ts"]) / 1_000_000.0, float(e["dur"]) / 1_000_000.0)\n                for e in cat_evts\n            ]\n            if bars:\n                ax.broken_barh(\n                    bars,\n                    (y_idx - 0.35, 0.7),\n                    facecolors=colors[cat],\n                    edgecolors="#0f172a",\n                    linewidth=0.5,\n                    alpha=0.92,\n                )\n\n        ax.axvline(panel_end_s, color="#f8fafc", linestyle="--", linewidth=1.3)\n        ax.text(\n            panel_end_s,\n            len(lane_order) - 0.45,\n            f" Done: {panel_end_s:.2f}s ",\n            color="#f8fafc",\n            fontsize=9.5,\n            fontweight="bold",\n            ha="right",\n            va="top",\n            bbox={"boxstyle": "round,pad=0.2", "facecolor": "#334155", "edgecolor": "#94a3b8"},\n        )\n        ax.set_yticks(range(len(lane_order)))\n        ax.set_yticklabels([lane_labels[c] for c in lane_order], color="#e2e8f0", fontsize=9.5)\n        ax.invert_yaxis()\n        ax.set_xlim(0, global_max_s * 1.04)\n        ax.set_title(title, color="#f8fafc", fontsize=11.5, fontweight="bold", loc="left", pad=8)\n        ax.grid(axis="x", color="#334155", linestyle=":", alpha=0.7)\n        ax.tick_params(colors="#cbd5e1")\n        for spine in ax.spines.values():\n            spine.set_color("#475569")\n\n    axes[-1].set_xlabel("Timeline (seconds)", color="#e2e8f0", fontsize=10.5)\n    legend_patches = [\n        mpatches.Patch(color=colors["GPU"], label="GPU Active Compute"),\n        mpatches.Patch(color=colors["BUBBLE"], label="GPU Bubble (Wasted Waiting Time)"),\n        mpatches.Patch(color=colors["CPU"], label="CPU (PNG Decode / Lanczos / H.264 MP4)"),\n        mpatches.Patch(color=colors["IO"], label="IO (Disk Read/Write & PCIe Model Offload)"),\n    ]\n    fig.legend(\n        handles=legend_patches,\n        loc="upper right",\n        ncol=2,\n        frameon=True,\n        facecolor="#1e293b",\n        edgecolor="#475569",\n        labelcolor="#f8fafc",\n        fontsize=8.5,\n    )\n    fig.suptitle(\n        "Bubbleless v0.1 — IO / CPU / GPU Timeline & Bubble Elimination",\n        color="#f8fafc",\n        fontsize=13,\n        fontweight="bold",\n        x=0.02,\n        ha="left",\n        y=0.98,\n    )\n    plt.tight_layout(rect=[0, 0, 1, 0.91])\n    out_path = pathlib.Path(output_png)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(out_path, dpi=160)\n    plt.close(fig)\n    return out_path\n', encoding='utf-8')
(ROOT / 'run_phase1_exp.py').write_text('#!/usr/bin/env python3\n"""Phase 1 (v0.1) A/B Experiment Runner: Serial Baseline vs Async IO/CPU Overlap."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport dataclasses\nimport pathlib\nimport sys\n\nfrom bubbleless.profiler import TriComponentProfiler\nfrom bubbleless.scheduler import PipelineExecutor, SchedulePolicy\nfrom bubbleless.workload_t4 import (\n    DiffusersT4VideoWorkload,\n    SyntheticT4Workload,\n    generate_test_source_clips,\n)\n\n\ndef print_ab_comparison(\n    prof_serial: TriComponentProfiler,\n    prof_overlap: TriComponentProfiler,\n    parity_ok: bool,\n    trace_serial_path: pathlib.Path,\n    trace_overlap_path: pathlib.Path,\n) -> None:\n    s_base = prof_serial.summary()\n    s_opt = prof_overlap.summary()\n\n    t_base = s_base["total_wall_s"]\n    t_opt = s_opt["total_wall_s"]\n    saved_s = t_base - t_opt\n    speedup_pct = (saved_s / t_base * 100.0) if t_base > 0 else 0.0\n    throughput_gain = (t_base / t_opt) if t_opt > 0 else 1.0\n\n    print("\\n" + "=" * 72)\n    print("  BUBBLELESS PHASE 1 (v0.1) A/B BENCHMARK SUMMARY")\n    print("=" * 72)\n    print(f"{\'Metric\':<30} | {\'Serial Baseline\':>16} | {\'Async Overlap\':>16}")\n    print("-" * 72)\n    print(f"{\'End-to-End Total Time (s)\':<30} | {t_base:>15.2f}s | {t_opt:>15.2f}s")\n    print(f"{\'GPU Active Compute (s)\':<30} | {s_base[\'gpu_busy_s\']:>15.2f}s | {s_opt[\'gpu_busy_s\']:>15.2f}s")\n    print(f"{\'GPU Utilization (%)\':<30} | {s_base[\'gpu_util_pct\']:>15.1f}% | {s_opt[\'gpu_util_pct\']:>15.1f}%")\n    print(f"{\'GPU Bubble Time (s)\':<30} | {s_base[\'gpu_bubble_s\']:>15.2f}s | {s_opt[\'gpu_bubble_s\']:>15.2f}s")\n    print(f"{\'GPU Bubble Ratio (%)\':<30} | {s_base[\'gpu_bubble_pct\']:>15.1f}% | {s_opt[\'gpu_bubble_pct\']:>15.1f}%")\n    print(f"{\'  ├─ Waiting on IO (s)\':<30} | {s_base[\'bubble_wait_io_s\']:>15.2f}s | {s_opt[\'bubble_wait_io_s\']:>15.2f}s")\n    print(f"{\'  └─ Waiting on CPU (s)\':<30} | {s_base[\'bubble_wait_cpu_s\']:>15.2f}s | {s_opt[\'bubble_wait_cpu_s\']:>15.2f}s")\n    print(f"{\'Peak Host RAM (MB)\':<30} | {s_base[\'peak_ram_mb\']:>14.0f}MB | {s_opt[\'peak_ram_mb\']:>14.0f}MB")\n    print(f"{\'Peak GPU VRAM (MB)\':<30} | {s_base[\'peak_vram_mb\']:>14.0f}MB | {s_opt[\'peak_vram_mb\']:>14.0f}MB")\n    print("-" * 72)\n    print(f"  Time Saved       : {saved_s:.2f} s ({speedup_pct:+.1f}% wall-time reduction, {throughput_gain:.2f}x throughput)")\n    print(f"  Lossless Check   : {\'PASS (100% identical frame SHA-256)\' if parity_ok else \'MISMATCH\'}")\n    print(f"  Perfetto Traces  :")\n    print(f"    1) Baseline -> {trace_serial_path}")\n    print(f"    2) Overlap  -> {trace_overlap_path}")\n    print("    (Open https://ui.perfetto.dev in browser and drag-and-drop both JSON files!)")\n\n    # Bottleneck diagnosis hint for Phase 2!\n    rem_io = s_opt["bubble_wait_io_s"]\n    rem_cpu = s_opt["bubble_wait_cpu_s"]\n    print("-" * 72)\n    if rem_io >= rem_cpu and rem_io > 0.1:\n        print(\n            f"  [Bottleneck Diagnosis -> Next Target]: Remaining GPU bubble is dominated by IO "\n            f"({rem_io:.2f}s, mostly PCIe Model Offload) -> Ready for Phase 2 Prefetch/Cache!"\n        )\n    elif rem_cpu > 0.1:\n        print(\n            f"  [Bottleneck Diagnosis -> Next Target]: Remaining GPU bubble is dominated by CPU "\n            f"({rem_cpu:.2f}s) -> Candidate for NVENC hardware video encode!"\n        )\n    else:\n        print(\n            "  [Bottleneck Diagnosis -> Next Target]: GPU utilization is saturated! "\n            "Next bottleneck is GPU compute itself (TeaCache / FP8)."\n        )\n    print("=" * 72 + "\\n")\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="Run Bubbleless Phase 1 A/B Experiment")\n    parser.add_argument("--num-tasks", type=int, default=5, help="Number of video tasks in batch queue")\n    parser.add_argument("--steps", type=int, default=12, help="Denoising steps per video task")\n    parser.add_argument(\n        "--work-dir",\n        type=str,\n        default="/content/bubbleless_exp" if pathlib.Path("/content").exists() else "./exp_workspace",\n        help="Local disk workspace (avoid Google Drive FUSE mount)",\n    )\n    parser.add_argument(\n        "--mock",\n        action="store_true",\n        help="Use calibrated synthetic T4 workload (for CPU-only dry-run verification)",\n    )\n    parser.add_argument(\n        "--no-offload",\n        action="store_true",\n        help="Keep all models on GPU VRAM instead of CPU<->GPU model offloading",\n    )\n    args = parser.parse_args()\n\n    work_dir = pathlib.Path(args.work_dir).resolve()\n    print(f"[Bubbleless] Preparing {args.num_tasks} deterministic test clips in {work_dir} ...")\n    base_specs = generate_test_source_clips(work_dir, num_tasks=args.num_tasks)\n    for sp in base_specs:\n        sp.num_inference_steps = args.steps\n\n    if args.mock:\n        print("[Bubbleless] Mode: Synthetic T4 Dry-Run (--mock)")\n        workload = SyntheticT4Workload()\n    else:\n        print("[Bubbleless] Mode: Real CUDA T4 Diffusers Video-to-Video (fp16)")\n        workload = DiffusersT4VideoWorkload(enable_offload=not args.no_offload)\n        workload.warmup(base_specs[0])\n\n    # Separate output folders for serial vs overlap\n    serial_specs = [\n        dataclasses.replace(sp, output_dir=work_dir / "out_serial") for sp in base_specs\n    ]\n    overlap_specs = [\n        dataclasses.replace(sp, output_dir=work_dir / "out_overlap") for sp in base_specs\n    ]\n\n    print("\\n>>> [1/2] Running Baseline: SERIAL Execution ...")\n    prof_serial = TriComponentProfiler("Phase1_Serial_Baseline")\n    exec_serial = PipelineExecutor(workload, SchedulePolicy(mode="serial"))\n    res_serial = exec_serial.run_batch(serial_specs, prof_serial)\n    print(prof_serial.format_report())\n\n    print("\\n>>> [2/2] Running Phase 1: ASYNC IO/CPU OVERLAP Execution ...")\n    prof_overlap = TriComponentProfiler("Phase1_Async_Overlap")\n    exec_overlap = PipelineExecutor(\n        workload,\n        SchedulePolicy(\n            mode="async_overlap",\n            max_prefetch_queue=2,\n            max_export_queue=2,\n            min_free_ram_mb=1500.0,\n        ),\n    )\n    res_overlap = exec_overlap.run_batch(overlap_specs, prof_overlap)\n    print(prof_overlap.format_report())\n\n    parity_ok = [r.frame_sha256 for r in res_serial] == [r.frame_sha256 for r in res_overlap]\n    trace_dir = work_dir / "traces"\n    t_serial = prof_serial.export_perfetto_json(trace_dir / "trace_phase1_serial.json")\n    t_overlap = prof_overlap.export_perfetto_json(trace_dir / "trace_phase1_overlap.json")\n\n    print_ab_comparison(prof_serial, prof_overlap, parity_ok, t_serial, t_overlap)\n\n    try:\n        from bubbleless.viz import plot_ab_swimlanes\n\n        chart_path = plot_ab_swimlanes(\n            t_serial, t_overlap, trace_dir / "timeline_phase1_comparison.png"\n        )\n        print(f"[Bubbleless] Saved visual swimlane timeline chart -> {chart_path}")\n    except ImportError:\n        pass\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n', encoding='utf-8')
print('Bubbleless v0.1 source files written to /content/bubbleless!')


### Step 3：运行 A/B 对比实验（串行 Baseline vs 异步双缓冲 Overlap）
默认连续跑 **5 个 16帧 AI 视频重绘任务**（开启 CPU<->GPU 模型卸载模拟消费级显存受限场景），并做 SHA-256 无损校验。


In [ ]:
%cd /content/bubbleless
!python3 run_phase1_exp.py --num-tasks 5 --steps 12 --work-dir /content/bubbleless_exp


### Step 4：直接在页面内查看【IO / CPU / GPU 气泡泳道对比图】并下载 Perfetto Trace


In [ ]:
from IPython.display import Image as IPyImage, display
import pathlib

trace_dir = pathlib.Path('/content/bubbleless_exp/traces')
display(IPyImage(filename=str(trace_dir / 'timeline_phase1_comparison.png')))

try:
    from google.colab import files
    print('正在触发下载 Perfetto 时间轴文件（可拖入 https://ui.perfetto.dev 放大查看细节）：')
    files.download(str(trace_dir / 'trace_phase1_serial.json'))
    files.download(str(trace_dir / 'trace_phase1_overlap.json'))
except Exception as e:
    print('Trace files saved at:', trace_dir)
